In [23]:
%pip install sentencepiece protobuf==3.20.3

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import torch

In [2]:
from transformers import AutoTokenizer, AutoModel, BertModel, BertTokenizer

# bert: bert-base-uncased,
# biobert: dmis-lab/biobert-base-cased-v1.1; use BertTokenizer and BertModel instead of AutoTokenizer and AutoModel
# chemberta v2: DeepChem/ChemBERTa-77M-MTR
model_name = "DeepChem/ChemBERTa-77M-MTR"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

model.eval()

c:\Users\Diya Budlakoti\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 53/53 [00:00<00:00, 26498.76it/s]
RobertaModel LOAD REPORT from: DeepChem/ChemBERTa-77M-MTR
Key                             | Status     | 
--------------------------------+------------+-
regression.out_proj.bias        | UNEXPECTED | 
regression.dense.weight         | UNEXPECTED | 
regression.out_proj.weight      | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
regression.dense.bias           | UNEXPECTED | 
norm_mean                       | UNEXPECTED | 
norm_std                        | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from diff

RobertaModel(
  (embeddings): RobertaEmbeddings(
    (word_embeddings): Embedding(600, 384, padding_idx=1)
    (token_type_embeddings): Embedding(1, 384)
    (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.144, inplace=False)
    (position_embeddings): Embedding(515, 384, padding_idx=1)
  )
  (encoder): RobertaEncoder(
    (layer): ModuleList(
      (0-2): 3 x RobertaLayer(
        (attention): RobertaAttention(
          (self): RobertaSelfAttention(
            (query): Linear(in_features=384, out_features=384, bias=True)
            (key): Linear(in_features=384, out_features=384, bias=True)
            (value): Linear(in_features=384, out_features=384, bias=True)
            (dropout): Dropout(p=0.109, inplace=False)
          )
          (output): RobertaSelfOutput(
            (dense): Linear(in_features=384, out_features=384, bias=True)
            (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
            (dropou

In [3]:
def get_embedding(smiles):
    inputs = tokenizer(
        smiles,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    )
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    # CLS token embedding (standard choice)
    embedding = outputs.last_hidden_state[:, 0, :]
    
    return embedding.squeeze(0)

In [4]:
def batch_embeddings(smiles_list, batch_size=32):
    all_embeddings = []
    
    for i in range(0, len(smiles_list), batch_size):
        batch = smiles_list[i:i+batch_size]
        
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=128
        )
        
        with torch.no_grad():
            outputs = model(**inputs)
        
        batch_emb = outputs.last_hidden_state[:, 0, :]
        all_embeddings.append(batch_emb)
    
    return torch.cat(all_embeddings, dim=0)

In [5]:
import pandas as pd

df = pd.read_csv("C:\\Users\\Diya Budlakoti\\Desktop\\SIN_proj\\dataset\\final_processed_data.csv")
unique_smiles = list(set(df["SMILES1"]) | set(df["SMILES2"]))

unique_embeddings = batch_embeddings(unique_smiles)

# Create mapping
smiles_to_emb = dict(zip(unique_smiles, unique_embeddings))
emb_s1 = torch.stack([smiles_to_emb[s] for s in df["SMILES1"]])
emb_s2 = torch.stack([smiles_to_emb[s] for s in df["SMILES2"]])

In [6]:
combined_embeddings = []

for e1, e2 in zip(emb_s1, emb_s2):
    combined_embeddings.append(e1 + e2)

combined_embeddings = torch.stack(combined_embeddings)

In [7]:
import ast

def clean_side_effects(x):
    if isinstance(x, str):
        x = ast.literal_eval(x)
    if isinstance(x, list) and len(x) == 1 and isinstance(x[0], list):
        x = x[0]
    return x

df["Side Effect Name"] = df["Side Effect Name"].apply(clean_side_effects)

print(df["Side Effect Name"].iloc[0])
print(type(df["Side Effect Name"].iloc[0]))

from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df["Side Effect Name"])

y = torch.tensor(y, dtype=torch.float32)

torch.save({
    "X": combined_embeddings,
    "y": y,
    "label_names": mlb.classes_
}, "chemBERTa_dataset.pt")

['body temperature increased', 'Blood Calcium Increased', 'heart attack', 'eruption', 'Drug hypersensitivity', 'pleural fibrosis', 'confusion', 'Infection Urinary Tract', 'muscle weakness', 'hyperuricaemia', 'Lymphocytes decreased', 'icterus', 'pyelonephritis', 'anaemia', 'adynamic ileus', 'Acute Respiratory Distress Syndrome', 'enterocolitis', 'cancer', 'Bleeding gums', 'nephrosclerosis', 'bladder retention', 'nausea', 'hypochloraemia', 'neoplasia', 'ild', 'blood sodium decreased', 'atrial flutter', 'hypokalaemia', 'Pleural Effusion', 'streptococcal infection', 'ascites', 'neumonia', 'angina', 'bundle branch block right', 'hyperglycaemia', 'arterial pressure NOS increased', 'edema', 'AFIB', 'Blood calcium decreased', 'aspergillosis', 'Fatigue', 'Spinal Compression Fracture', 'aspiration pneumonia', 'squamous cell carcinoma of skin', 'diarrhea', 'dysuria', 'osteomyelitis', 'gastrointestinal bleed', 'lung edema', 'Cardiac ischemia', 'lung fibrosis', 'myoglobinuria', 'coma', 'amyotrophy'

In [30]:
embedding_cache = {}
def get_embedding_cached(smiles):
    if smiles in embedding_cache:
        return embedding_cache[smiles]
    
    emb = get_embedding(smiles)
    embedding_cache[smiles] = emb
    return emb

In [31]:
for s in df["SMILES1"]:
    get_embedding_cached(s)

for s in df["SMILES2"]:
    get_embedding_cached(s)
print(len(embedding_cache))

644


In [32]:
torch.save(embedding_cache, "chemberta_smiles_embeddings.pt")